In [1]:
from ultralytics import YOLO

import torch
import torchvision.transforms as transforms

from torchvision.models import resnet18
import torchvision.models as models
from PIL import Image

import cv2
import numpy as np
import pandas as pd
from pathlib import Path
import shutil
from tqdm import tqdm
import time

In [2]:
YOLO_MODEL = YOLO("C:\\Users\\klanz\\Desktop\\MAGISTERKA\\notebooks\\runs\\detect\\runs\\YOLOv8_baseline-2\\weights\\best.pt")

c:\Users\klanz\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

cnn = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
num_features = cnn.fc.in_features
cnn.fc = torch.nn.Sequential(
    torch.nn.Linear(num_features, 128),
    torch.nn.ReLU(),
    torch.nn.Dropout(0.5),
    torch.nn.Linear(128, 2)
)

cnn.load_state_dict(torch.load("C:\\Users\\klanz\\Desktop\\MAGISTERKA\\notebooks\\best_chicken_cnn_augument2.pth"))

cnn.to(DEVICE)

cnn.eval()

C:\Users\klanz\AppData\Local\Temp\ipykernel_41540\102316409.py:12: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  cnn.load_state_dict(torch.load("C:\\Users\\klanz\\Desktop\\M

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

In [4]:
transform = transforms.Compose([

    transforms.Resize((224,224)),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485,0.456,0.406],
        std=[0.229,0.224,0.225]
    )

])

In [5]:
def classify_crop(crop):

    image = Image.fromarray(cv2.cvtColor(crop,cv2.COLOR_BGR2RGB))

    tensor = transform(image)

    tensor = tensor.unsqueeze(0)

    tensor = tensor.to(DEVICE)

    with torch.no_grad():

        prediction = cnn(tensor)

        prediction = prediction.argmax(1).item()

    return prediction

In [6]:
def door_decision(predictions):

    if len(predictions) == 0:
        return "CLOSE"

    cnn_classes = [p["cnn_prediction"] for p in predictions]

    if 1 in cnn_classes:
        return "CLOSE"

    if 0 in cnn_classes:
        return "OPEN"

    return "CLOSE"

In [7]:
def door_decision_yolo(predictions):

    if len(predictions) == 0:
        return "CLOSE"

    classes = [p["class"] for p in predictions]

    if 1 in classes:
        return "CLOSE"

    if 0 in classes:
        return "OPEN"

    return "CLOSE"

In [8]:
CLASS_NAMES = {
    0: "chicken",
    1: "not_chicken"
}

In [9]:
def run_yolo(image_path, conf=0.5):

    result = YOLO_MODEL.predict(
        source=str(image_path),
        conf=conf,
        verbose=False
    )[0]

    predictions = []

    for box in result.boxes:

        cls = int(box.cls.item())

        confidence = float(box.conf.item())

        x1, y1, x2, y2 = box.xyxy.cpu().numpy()[0]

        predictions.append({

            "class": cls,
            "class_name": CLASS_NAMES[cls],
            "confidence": confidence,
            "bbox": [int(x1), int(y1), int(x2), int(y2)]

        })

    return predictions

In [10]:
def run_pipeline(image_path, conf=0.5):

    detections = run_yolo(image_path, conf)

    image = cv2.imread(str(image_path))

    final_predictions = []

    for det in detections:

        x1, y1, x2, y2 = det["bbox"]

        crop = image[y1:y2, x1:x2]

        if crop.size == 0:
            continue

        cnn_prediction = classify_crop(crop)

        final_predictions.append({

            "yolo_prediction": det["class"],

            "cnn_prediction": cnn_prediction,

            "confidence": det["confidence"],

            "bbox": det["bbox"]

        })

    return final_predictions

In [11]:
def load_ground_truth(label_path):

    if not Path(label_path).exists():
        return []

    gt=[]

    with open(label_path) as f:

        for line in f:

            line=line.strip()

            if line=="":

                continue

            cls=int(line.split()[0])

            gt.append(cls)

    return gt

In [12]:
def ground_truth_decision(gt_classes):

    if 1 in gt_classes:
        return "CLOSE"

    if 0 in gt_classes:
        return "OPEN"

    return "CLOSE"

In [13]:
test_images = sorted(Path("C:\\Users\\klanz\\Desktop\\MAGISTERKA\\dataset\\test\\images").glob("*"))

results = []

In [14]:
for image_path in test_images:

    label_path = Path(r"C:\Users\klanz\Desktop\MAGISTERKA\dataset\test\labels") / (image_path.stem + ".txt")

    gt = load_ground_truth(label_path)

    gt_decision = ground_truth_decision(gt)

    # ---------- YOLO ----------

    start = time.perf_counter()

    yolo_predictions = run_yolo(image_path)

    yolo_time = (time.perf_counter()-start)*1000

    yolo_decision = door_decision_yolo(yolo_predictions)

    # ---------- PIPELINE ----------

    start = time.perf_counter()

    pipeline_predictions = run_pipeline(image_path)

    pipeline_time = (time.perf_counter()-start)*1000

    pipeline_decision = door_decision(pipeline_predictions)

    results.append({

        "image": image_path.name,

        "ground_truth": gt_decision,

        "yolo": yolo_decision,

        "pipeline": pipeline_decision,

        "yolo_time_ms": yolo_time,

        "pipeline_time_ms": pipeline_time,

        "objects_gt": gt,

        "objects_yolo": [x["class"] for x in yolo_predictions],

        "objects_pipeline": [x["cnn_prediction"] for x in pipeline_predictions]

    })

In [15]:
df = pd.DataFrame(results)

df.head(20)

,image,ground_truth,yolo,pipeline,yolo_time_ms,pipeline_time_ms,objects_gt,objects_yolo,objects_pipeline
0,1000.jpeg,OPEN,OPEN,OPEN,397.6308,43.7075,"[0, 0]",[0],[0]
1,1035.jpeg,OPEN,OPEN,OPEN,17.5309,18.1925,[0],[0],[0]
2,1048.jpeg,OPEN,OPEN,OPEN,11.0565,15.8235,[0],[0],[0]
3,1053.jpeg,OPEN,OPEN,CLOSE,12.1608,22.6138,"[0, 0, 0, 0, 0]","[0, 0, 0]","[0, 0, 1]"
4,1054.jpeg,OPEN,OPEN,OPEN,12.4042,25.6462,"[0, 0, 0]","[0, 0, 0]","[0, 0, 0]"
5,1085.jpeg,OPEN,OPEN,OPEN,15.7630,17.1039,[0],[0],[0]
6,109.jpeg,OPEN,OPEN,OPEN,13.5736,16.5425,[0],[0],[0]
7,110.jpeg,OPEN,OPEN,OPEN,11.3021,15.4602,[0],[0],[0]
8,1108.jpeg,OPEN,OPEN,OPEN,11.2318,16.3553,[0],[0],[0]
9,1133.jpeg,OPEN,OPEN,CLOSE,11.3869,24.6853,"[0, 0, 0]","[0, 0, 0]","[0, 0, 1]"


In [16]:
from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import f1_score
from sklearn.metrics import confusion_matrix

import numpy as np

In [17]:
decision_map = {

    "OPEN":1,

    "CLOSE":0

}

In [18]:
def evaluate_system(df, prediction_column, time_column):

    gt = df["ground_truth"].map(decision_map)

    pred = df[prediction_column].map(decision_map)

    accuracy = accuracy_score(gt, pred)

    precision = precision_score(
        gt,
        pred,
        zero_division=0
    )

    recall = recall_score(
        gt,
        pred,
        zero_division=0
    )

    f1 = f1_score(
        gt,
        pred,
        zero_division=0
    )

    cm = confusion_matrix(gt, pred)

    tn, fp, fn, tp = cm.ravel()

    avg_time = df[time_column].mean()

    print("="*50)

    print(prediction_column)

    print("="*50)

    print(f"Accuracy : {accuracy:.4f}")

    print(f"Precision: {precision:.4f}")

    print(f"Recall   : {recall:.4f}")

    print(f"F1-score : {f1:.4f}")

    print()

    print(f"TP : {tp}")

    print(f"FP : {fp}")

    print(f"TN : {tn}")

    print(f"FN : {fn}")

    print()

    print(f"Średni czas: {avg_time:.2f} ms")

    return {

        "Accuracy":accuracy,

        "Precision":precision,

        "Recall":recall,

        "F1":f1,

        "TP":tp,

        "FP":fp,

        "TN":tn,

        "FN":fn,

        "Time":avg_time

    }

In [19]:
yolo_results = evaluate_system(

    df,

    "yolo",

    "yolo_time_ms"

)

yolo
Accuracy : 0.9886
Precision: 0.9918
Recall   : 0.9877
F1-score : 0.9897

TP : 481
FP : 4
TN : 387
FN : 6

Średni czas: 14.26 ms


In [20]:
pipeline_results = evaluate_system(

    df,

    "pipeline",

    "pipeline_time_ms"

)

pipeline
Accuracy : 0.9806
Precision: 0.9855
Recall   : 0.9795
F1-score : 0.9825

TP : 477
FP : 7
TN : 384
FN : 10

Średni czas: 21.39 ms


In [21]:
comparison = pd.DataFrame(

    [

        yolo_results,

        pipeline_results

    ],

    index=[

        "YOLO",

        "YOLO + CNN"

    ]

)

comparison

,Accuracy,Precision,Recall,F1,TP,FP,TN,FN,Time
YOLO,0.988610,0.991753,0.987680,0.989712,481,4,387,6,14.258203
YOLO + CNN,0.980638,0.985537,0.979466,0.982492,477,7,384,10,21.391188


In [22]:
dangerous_yolo = df[

    (df["ground_truth"]=="CLOSE") &

    (df["yolo"]=="OPEN")

]

In [23]:
dangerous_pipeline = df[

    (df["ground_truth"]=="CLOSE") &

    (df["pipeline"]=="OPEN")

]

In [24]:
print()

print("Krytyczne błędy")

print("----------------")

print("YOLO:",len(dangerous_yolo))

print("YOLO+CNN:",len(dangerous_pipeline))


Krytyczne błędy
----------------
YOLO: 4
YOLO+CNN: 7


In [25]:
improved = df[

    (df["ground_truth"]=="CLOSE") &

    (df["yolo"]=="OPEN") &

    (df["pipeline"]=="CLOSE")

]

improved

,image,ground_truth,yolo,pipeline,yolo_time_ms,pipeline_time_ms,objects_gt,objects_yolo,objects_pipeline
327,Image-82-7d6856.jpg,CLOSE,OPEN,CLOSE,13.4512,18.2304,[1],[0],[1]
333,Image-84-bd2f1b.jpg,CLOSE,OPEN,CLOSE,10.7843,14.1846,[1],[0],[1]


In [26]:
empty_images = 0
false_detections = 0

for row in results:

    if len(row["objects_gt"]) == 0:

        empty_images += 1

        if len(row["objects_yolo"]) > 0:
            false_detections += 1

print("Puste obrazy:",empty_images)
print("Fałszywe detekcje:",false_detections)

if empty_images>0:

    print(
        "Odsetek:",
        false_detections/empty_images
    )

Puste obrazy: 37
Fałszywe detekcje: 2
Odsetek: 0.05405405405405406


In [27]:
fp_images=[]

for row in results:

    if row["ground_truth"]=="CLOSE" and row["pipeline"]=="OPEN":

        fp_images.append(row["image"])

print("False Positive:",len(fp_images))

fp_images

False Positive: 7


['coyote__lila_WSU_Lynx_IMG_0965_jpg.rf.vqQg0CptK1hvCGBkRVla.jpg',
 'Image-52-fd6d74.jpg',
 'Image-88-8f30f6.jpg',
 'raptor__gbif_tx_raptor_0203_jpg.rf.YVU6MioP47sA51XvItUr.jpg',
 'raptor__raptor_028_jpg.rf.a9tIjTTL30oWJULOxUVa.jpg',
 'raptor__raptor_043_jpg.rf.Zvj9EKGgySzsjUZnOEDi.jpg',
 'raptor__raptor_050_jpg.rf.duuf3tqrVk3ZzBLkisY3.jpg']

In [28]:
fn_images=[]

for row in results:

    if row["ground_truth"]=="OPEN" and row["pipeline"]=="CLOSE":

        fn_images.append(row["image"])

print("False Negative:",len(fn_images))

fn_images

False Negative: 10


['1053.jpeg',
 '1133.jpeg',
 'neg_poultry__poultry_115_jpg.rf.n5StsXV8mhnHSm7Os8df.jpg',
 'neg_poultry__poultry_133_jpg.rf.Y9pO5YIarKaSPg8D9A7C.jpg',
 'OIP-aoDZ9NQTzSJ5RInD_vwARgHaFj.jpeg',
 'OIP-lzNLIe_vySMcitY9uEqapgHaFj.jpeg',
 'OIP-NEf6Uuinf5Z2jTEDnPLUHwHaFj.jpeg',
 'OIP-R1UhJyfXM5EoZdTAA2WzygHaFj.jpeg',
 'OIP-RtkErMx6_ECG2NR4KTQiTQHaE8.jpeg',
 'OIP-XHrr9AeiI8Egrt85dSx7mwHaFj.jpeg']

In [29]:
fp_yolo=[]

for row in results:

    if row["ground_truth"]=="CLOSE" and row["yolo"]=="OPEN":

        fp_yolo.append(row["image"])

len(fp_yolo)

fp_yolo

['Image-82-7d6856.jpg',
 'Image-84-bd2f1b.jpg',
 'Image-88-8f30f6.jpg',
 'raptor__raptor_028_jpg.rf.a9tIjTTL30oWJULOxUVa.jpg']

In [30]:
fn_yolo=[]

for row in results:

    if row["ground_truth"]=="OPEN" and row["yolo"]=="CLOSE":

        fn_yolo.append(row["image"])

len(fn_yolo)

fn_yolo

['neg_poultry__poultry_244_jpg.rf.X1DkC0GjdbMdNEwCaauz.jpg',
 'OIP-aoDZ9NQTzSJ5RInD_vwARgHaFj.jpeg',
 'OIP-NEf6Uuinf5Z2jTEDnPLUHwHaFj.jpeg',
 'OIP-OLDB-DMfG6VSQCZwAyo_rAHaFj.jpeg',
 'OIP-R1UhJyfXM5EoZdTAA2WzygHaFj.jpeg',
 'OIP-RtkErMx6_ECG2NR4KTQiTQHaE8.jpeg']

In [31]:

OUTPUT = Path(r"C:\Users\klanz\Desktop\MAGISTERKA\dataset_experiments")



In [32]:
image_path_dark = sorted(Path("C:\\Users\\klanz\\Desktop\\MAGISTERKA\\dataset_experiments\\dark\\images").glob("*"))
image_path_night = sorted(Path("C:\\Users\\klanz\\Desktop\\MAGISTERKA\\dataset_experiments\\night\\images").glob("*"))
image_path_occlusion = sorted(Path("C:\\Users\\klanz\\Desktop\\MAGISTERKA\\dataset_experiments\\occlusion\\images").glob("*"))
image_path_motion_blur = sorted(Path("C:\\Users\\klanz\\Desktop\\MAGISTERKA\\dataset_experiments\\motion_blur\\images").glob("*"))

results_dark = []
results_night = []
results_occlusion = []
results_motion_blur = []


In [33]:
for image_path in image_path_dark:

    label_path = Path(r"C:\Users\klanz\Desktop\MAGISTERKA\dataset\test\labels") / (image_path.stem + ".txt")
    gt = load_ground_truth(label_path)
    gt_decision = ground_truth_decision(gt)

    # ---------- YOLO ----------

    start = time.perf_counter()
    yolo_predictions = run_yolo(image_path)
    yolo_time = (time.perf_counter()-start)*1000
    yolo_decision = door_decision_yolo(yolo_predictions)

    # ---------- PIPELINE ----------

    start = time.perf_counter()
    pipeline_predictions = run_pipeline(image_path)
    pipeline_time = (time.perf_counter()-start)*1000
    pipeline_decision = door_decision(pipeline_predictions)

    results_dark.append({

        "image": image_path.name,
        "ground_truth": gt_decision,
        "yolo": yolo_decision,
        "pipeline": pipeline_decision,
        "yolo_time_ms": yolo_time,
        "pipeline_time_ms": pipeline_time,
        "objects_gt": gt,
        "objects_yolo": [x["class"] for x in yolo_predictions],
        "objects_pipeline": [x["cnn_prediction"] for x in pipeline_predictions]

    })

In [34]:
for image_path in image_path_night:

    label_path = Path(r"C:\Users\klanz\Desktop\MAGISTERKA\dataset\test\labels") / (image_path.stem + ".txt")
    gt = load_ground_truth(label_path)
    gt_decision = ground_truth_decision(gt)

    # ---------- YOLO ----------

    start = time.perf_counter()
    yolo_predictions = run_yolo(image_path)
    yolo_time = (time.perf_counter()-start)*1000
    yolo_decision = door_decision_yolo(yolo_predictions)

    # ---------- PIPELINE ----------

    start = time.perf_counter()
    pipeline_predictions = run_pipeline(image_path)
    pipeline_time = (time.perf_counter()-start)*1000
    pipeline_decision = door_decision(pipeline_predictions)

    results_night.append({

        "image": image_path.name,
        "ground_truth": gt_decision,
        "yolo": yolo_decision,
        "pipeline": pipeline_decision,
        "yolo_time_ms": yolo_time,
        "pipeline_time_ms": pipeline_time,
        "objects_gt": gt,
        "objects_yolo": [x["class"] for x in yolo_predictions],
        "objects_pipeline": [x["cnn_prediction"] for x in pipeline_predictions]

    })

In [35]:
for image_path in image_path_occlusion:

    label_path = Path(r"C:\Users\klanz\Desktop\MAGISTERKA\dataset\test\labels") / (image_path.stem + ".txt")
    gt = load_ground_truth(label_path)
    gt_decision = ground_truth_decision(gt)

    # ---------- YOLO ----------

    start = time.perf_counter()
    yolo_predictions = run_yolo(image_path)
    yolo_time = (time.perf_counter()-start)*1000
    yolo_decision = door_decision_yolo(yolo_predictions)

    # ---------- PIPELINE ----------

    start = time.perf_counter()
    pipeline_predictions = run_pipeline(image_path)
    pipeline_time = (time.perf_counter()-start)*1000
    pipeline_decision = door_decision(pipeline_predictions)

    results_occlusion.append({
        "image": image_path.name,
        "ground_truth": gt_decision,
        "yolo": yolo_decision,
        "pipeline": pipeline_decision,
        "yolo_time_ms": yolo_time,
        "pipeline_time_ms": pipeline_time,
        "objects_gt": gt,
        "objects_yolo": [x["class"] for x in yolo_predictions],
        "objects_pipeline": [x["cnn_prediction"] for x in pipeline_predictions]

    })

In [36]:
for image_path in image_path_motion_blur:

    label_path = Path(r"C:\Users\klanz\Desktop\MAGISTERKA\dataset\test\labels") / (image_path.stem + ".txt")
    gt = load_ground_truth(label_path)
    gt_decision = ground_truth_decision(gt)

    # ---------- YOLO ----------

    start = time.perf_counter()
    yolo_predictions = run_yolo(image_path)
    yolo_time = (time.perf_counter()-start)*1000
    yolo_decision = door_decision_yolo(yolo_predictions)

    # ---------- PIPELINE ----------

    start = time.perf_counter()
    pipeline_predictions = run_pipeline(image_path)
    pipeline_time = (time.perf_counter()-start)*1000
    pipeline_decision = door_decision(pipeline_predictions)

    results_motion_blur.append({

        "image": image_path.name,
        "ground_truth": gt_decision,
        "yolo": yolo_decision,
        "pipeline": pipeline_decision,
        "yolo_time_ms": yolo_time,
        "pipeline_time_ms": pipeline_time,
        "objects_gt": gt,
        "objects_yolo": [x["class"] for x in yolo_predictions],
        "objects_pipeline": [x["cnn_prediction"] for x in pipeline_predictions]

    })

In [37]:
df_dark = pd.DataFrame(results_dark)
df_night = pd.DataFrame(results_night)
df_occlusion = pd.DataFrame(results_occlusion)
df_motion_blur = pd.DataFrame(results_motion_blur)

df_dark.head(10)


,image,ground_truth,yolo,pipeline,yolo_time_ms,pipeline_time_ms,objects_gt,objects_yolo,objects_pipeline
0,1000.jpeg,OPEN,CLOSE,CLOSE,13.5866,13.5153,"[0, 0]",[],[]
1,1035.jpeg,OPEN,OPEN,OPEN,13.2272,18.7740,[0],[0],[0]
2,1048.jpeg,OPEN,OPEN,OPEN,13.3296,17.3549,[0],[0],[0]
3,1053.jpeg,OPEN,OPEN,OPEN,12.8668,17.7806,"[0, 0, 0, 0, 0]",[0],[0]
4,1054.jpeg,OPEN,OPEN,OPEN,13.0738,22.8535,"[0, 0, 0]","[0, 0, 0]","[0, 0, 0]"
5,1085.jpeg,OPEN,OPEN,CLOSE,12.8537,17.1302,[0],[0],[1]
6,109.jpeg,OPEN,OPEN,OPEN,12.6518,22.4131,[0],[0],[0]
7,110.jpeg,OPEN,OPEN,OPEN,12.3048,17.0874,[0],[0],[0]
8,1108.jpeg,OPEN,OPEN,OPEN,12.5722,16.9337,[0],[0],[0]
9,1133.jpeg,OPEN,OPEN,OPEN,12.1440,24.4651,"[0, 0, 0]","[0, 0, 0]","[0, 0, 0]"


In [38]:
df_night.head(10)

,image,ground_truth,yolo,pipeline,yolo_time_ms,pipeline_time_ms,objects_gt,objects_yolo,objects_pipeline
0,1000.jpeg,OPEN,CLOSE,CLOSE,14.4518,11.2022,"[0, 0]",[],[]
1,1035.jpeg,OPEN,OPEN,OPEN,9.9065,14.0160,[0],[0],[0]
2,1048.jpeg,OPEN,OPEN,OPEN,10.9587,15.2274,[0],[0],[0]
3,1053.jpeg,OPEN,OPEN,OPEN,10.0688,14.1858,"[0, 0, 0, 0, 0]",[0],[0]
4,1054.jpeg,OPEN,OPEN,CLOSE,12.4219,21.4062,"[0, 0, 0]","[0, 0, 0]","[1, 0, 0]"
5,1085.jpeg,OPEN,OPEN,CLOSE,11.1381,16.0991,[0],[0],[1]
6,109.jpeg,OPEN,OPEN,OPEN,11.1231,15.9558,[0],[0],[0]
7,110.jpeg,OPEN,OPEN,OPEN,10.5883,15.8073,[0],[0],[0]
8,1108.jpeg,OPEN,OPEN,OPEN,12.1665,15.3169,[0],[0],[0]
9,1133.jpeg,OPEN,OPEN,OPEN,10.4830,18.7497,"[0, 0, 0]","[0, 0]","[0, 0]"


In [39]:
df_occlusion.head(10)


,image,ground_truth,yolo,pipeline,yolo_time_ms,pipeline_time_ms,objects_gt,objects_yolo,objects_pipeline
0,1000.jpeg,OPEN,CLOSE,CLOSE,13.8801,11.3661,"[0, 0]",[],[]
1,1035.jpeg,OPEN,OPEN,OPEN,11.2434,19.9777,[0],[0],[0]
2,1048.jpeg,OPEN,OPEN,OPEN,10.7606,16.2701,[0],[0],[0]
3,1053.jpeg,OPEN,OPEN,CLOSE,11.9361,22.5663,"[0, 0, 0, 0, 0]","[0, 0, 0]","[0, 1, 0]"
4,1054.jpeg,OPEN,OPEN,OPEN,11.4714,19.5448,"[0, 0, 0]","[0, 0]","[0, 0]"
5,1085.jpeg,OPEN,CLOSE,CLOSE,11.4645,11.3389,[0],[],[]
6,109.jpeg,OPEN,OPEN,OPEN,9.9018,15.1052,[0],[0],[0]
7,110.jpeg,OPEN,OPEN,OPEN,12.6616,15.3053,[0],[0],[0]
8,1108.jpeg,OPEN,OPEN,OPEN,10.7227,15.1164,[0],[0],[0]
9,1133.jpeg,OPEN,OPEN,CLOSE,10.5351,22.0550,"[0, 0, 0]","[0, 0, 0]","[0, 0, 1]"


In [40]:
df_motion_blur.head(10)

,image,ground_truth,yolo,pipeline,yolo_time_ms,pipeline_time_ms,objects_gt,objects_yolo,objects_pipeline
0,1000.jpeg,OPEN,OPEN,OPEN,15.0272,16.6658,"[0, 0]",[0],[0]
1,1035.jpeg,OPEN,OPEN,OPEN,11.2007,14.9860,[0],[0],[0]
2,1048.jpeg,OPEN,OPEN,OPEN,10.4092,15.4003,[0],[0],[0]
3,1053.jpeg,OPEN,OPEN,OPEN,11.3990,15.9029,"[0, 0, 0, 0, 0]",[0],[0]
4,1054.jpeg,OPEN,CLOSE,CLOSE,11.4791,16.1709,"[0, 0, 0]",[1],[1]
5,1085.jpeg,OPEN,OPEN,CLOSE,10.3782,14.8527,[0],[0],[1]
6,109.jpeg,OPEN,OPEN,OPEN,10.7475,13.8461,[0],[0],[0]
7,110.jpeg,OPEN,OPEN,OPEN,10.1587,16.6509,[0],[0],[0]
8,1108.jpeg,OPEN,OPEN,OPEN,12.8250,15.5248,[0],[0],[0]
9,1133.jpeg,OPEN,OPEN,OPEN,10.7090,14.2927,"[0, 0, 0]",[0],[0]


In [41]:
yolo_results1 = evaluate_system(
    df_dark,
    "yolo",
    "yolo_time_ms"
)
pipeline_results1 = evaluate_system(
    df_dark,
    "pipeline",
    "pipeline_time_ms"
)

yolo
Accuracy : 0.9795
Precision: 0.9896
Recall   : 0.9733
F1-score : 0.9814

TP : 474
FP : 5
TN : 386
FN : 13

Średni czas: 11.75 ms
pipeline
Accuracy : 0.9692
Precision: 0.9915
Recall   : 0.9528
F1-score : 0.9717

TP : 464
FP : 4
TN : 387
FN : 23

Średni czas: 18.72 ms


In [42]:
yolo_results2 = evaluate_system(
    df_night,
    "yolo",
    "yolo_time_ms"
)
pipeline_results2 = evaluate_system(
    df_night,
    "pipeline",
    "pipeline_time_ms"
)

yolo
Accuracy : 0.9658
Precision: 0.9872
Recall   : 0.9507
F1-score : 0.9686

TP : 463
FP : 6
TN : 385
FN : 24

Średni czas: 11.63 ms
pipeline
Accuracy : 0.9317
Precision: 0.9931
Recall   : 0.8830
F1-score : 0.9348

TP : 430
FP : 3
TN : 388
FN : 57

Średni czas: 18.14 ms


In [43]:
yolo_results3 = evaluate_system(
    df_occlusion,
    "yolo",
    "yolo_time_ms"
)
pipeline_results3 = evaluate_system(
    df_occlusion,
    "pipeline",
    "pipeline_time_ms"
)

yolo
Accuracy : 0.9487
Precision: 0.9911
Recall   : 0.9158
F1-score : 0.9520

TP : 446
FP : 4
TN : 387
FN : 41

Średni czas: 12.44 ms
pipeline
Accuracy : 0.9294
Precision: 0.9712
Recall   : 0.8994
F1-score : 0.9339

TP : 438
FP : 13
TN : 378
FN : 49

Średni czas: 19.95 ms


In [44]:
yolo_results4 = evaluate_system(
    df_motion_blur,
    "yolo",
    "yolo_time_ms"
)
pipeline_results4 = evaluate_system(
    df_motion_blur,
    "pipeline",
    "pipeline_time_ms"
)

yolo
Accuracy : 0.9180
Precision: 0.9882
Recall   : 0.8624
F1-score : 0.9211

TP : 420
FP : 5
TN : 386
FN : 67

Średni czas: 11.84 ms
pipeline
Accuracy : 0.8986
Precision: 0.9716
Recall   : 0.8419
F1-score : 0.9021

TP : 410
FP : 12
TN : 379
FN : 77

Średni czas: 17.62 ms


In [45]:
comparison = pd.DataFrame(
    [
        yolo_results1,
        pipeline_results1
    ],
    index=[
        "YOLO",
        "YOLO + CNN"
    ]
)
comparison

,Accuracy,Precision,Recall,F1,TP,FP,TN,FN,Time
YOLO,0.979499,0.989562,0.973306,0.981366,474,5,386,13,11.749530
YOLO + CNN,0.969248,0.991453,0.952772,0.971728,464,4,387,23,18.717257


In [46]:
comparison = pd.DataFrame(
    [
        yolo_results2,
        pipeline_results2

    ],
    index=[
        "YOLO",
        "YOLO + CNN"
    ]
)
comparison

,Accuracy,Precision,Recall,F1,TP,FP,TN,FN,Time
YOLO,0.965831,0.987207,0.950719,0.968619,463,6,385,24,11.626247
YOLO + CNN,0.931663,0.993072,0.882957,0.934783,430,3,388,57,18.142294


In [47]:
comparison = pd.DataFrame(
    [
        yolo_results3,
        pipeline_results3
    ],
    index=[
        "YOLO",
        "YOLO + CNN"
    ]
)
comparison

,Accuracy,Precision,Recall,F1,TP,FP,TN,FN,Time
YOLO,0.948747,0.991111,0.915811,0.951974,446,4,387,41,12.440679
YOLO + CNN,0.929385,0.971175,0.899384,0.933902,438,13,378,49,19.951463


In [48]:
comparison = pd.DataFrame(
    [
        yolo_results4,
        pipeline_results4
    ],
    index=[
        "YOLO",
        "YOLO + CNN"
    ]
)
comparison

,Accuracy,Precision,Recall,F1,TP,FP,TN,FN,Time
YOLO,0.917995,0.988235,0.862423,0.921053,420,5,386,67,11.844712
YOLO + CNN,0.898633,0.971564,0.841889,0.902090,410,12,379,77,17.623397


In [49]:
dangerous_yolo1 = df_dark[
    (df_dark["ground_truth"]=="CLOSE") &
    (df_dark["yolo"]=="OPEN")
]

In [50]:
dangerous_yolo2 = df_night[
    (df_night["ground_truth"]=="CLOSE") &
    (df_night["yolo"]=="OPEN")
]

In [51]:
dangerous_yolo3 = df_occlusion[
    (df_occlusion["ground_truth"]=="CLOSE") &
    (df_occlusion["yolo"]=="OPEN")
]

In [52]:
dangerous_yolo4 = df_motion_blur[
    (df_motion_blur["ground_truth"]=="CLOSE") &
    (df_motion_blur["yolo"]=="OPEN")
]

In [53]:
dangerous_pipeline1 = df_dark[
    (df_dark["ground_truth"]=="CLOSE") &
    (df_dark["pipeline"]=="OPEN")
]

In [54]:
dangerous_pipeline2 = df_night[
    (df_night["ground_truth"]=="CLOSE") &
    (df_night["pipeline"]=="OPEN")
]

In [55]:
dangerous_pipeline3 = df_occlusion[
    (df_occlusion["ground_truth"]=="CLOSE") &
    (df_occlusion["pipeline"]=="OPEN")
]

In [56]:
dangerous_pipeline4 = df_motion_blur[
    (df_motion_blur["ground_truth"]=="CLOSE") &
    (df_motion_blur["pipeline"]=="OPEN")
]

In [73]:
dangerous_both = df[
    (df["ground_truth"]=="CLOSE") & (df["yolo"]=="OPEN") & (df["pipeline"]=="OPEN")
]
dangerous_both1 = df_dark[
    (df_dark["ground_truth"]=="CLOSE") & (df_dark["yolo"]=="OPEN") & (df_dark["pipeline"]=="OPEN")
]
dangerous_both2 = df_night[
    (df_night["ground_truth"]=="CLOSE") & (df_night["yolo"]=="OPEN") & (df_night["pipeline"]=="OPEN")
]
dangerous_both3 = df_occlusion[
    (df_occlusion["ground_truth"]=="CLOSE") & (df_occlusion["yolo"]=="OPEN") & (df_occlusion["pipeline"]=="OPEN")
]
dangerous_both4 = df_motion_blur[
    (df_motion_blur["ground_truth"]=="CLOSE") & (df_motion_blur["yolo"]=="OPEN") & (df_motion_blur["pipeline"]=="OPEN")
]


In [58]:
comparison = pd.DataFrame(

    [   

        yolo_results,

        pipeline_results,

        yolo_results1,

        pipeline_results1,

        yolo_results2,

        pipeline_results2,

        yolo_results3,

        pipeline_results3,

        yolo_results4,

        pipeline_results4

    ],

    index=[

        "YOLO normal",
        "YOLO+CNN normal",
        "YOLO dark",
        "YOLO+CNN dark",
        "YOLO night",
        "YOLO+CNN night",
        "YOLO occlusion",
        "YOLO+CNN occlusion",
        "YOLO motion",
        "YOLO+CNN motion"

    ]

)

comparison

,Accuracy,Precision,Recall,F1,TP,FP,TN,FN,Time
YOLO normal,0.988610,0.991753,0.987680,0.989712,481,4,387,6,14.258203
YOLO+CNN normal,0.980638,0.985537,0.979466,0.982492,477,7,384,10,21.391188
YOLO dark,0.979499,0.989562,0.973306,0.981366,474,5,386,13,11.749530
YOLO+CNN dark,0.969248,0.991453,0.952772,0.971728,464,4,387,23,18.717257
YOLO night,0.965831,0.987207,0.950719,0.968619,463,6,385,24,11.626247
YOLO+CNN night,0.931663,0.993072,0.882957,0.934783,430,3,388,57,18.142294
YOLO occlusion,0.948747,0.991111,0.915811,0.951974,446,4,387,41,12.440679
YOLO+CNN occlusion,0.929385,0.971175,0.899384,0.933902,438,13,378,49,19.951463
YOLO motion,0.917995,0.988235,0.862423,0.921053,420,5,386,67,11.844712
YOLO+CNN motion,0.898633,0.971564,0.841889,0.902090,410,12,379,77,17.623397


In [59]:
print()
print("Krytyczne błędy, wpuszczenie drapieżnika")
print("----------------")
print("YOLO dark:",len(dangerous_yolo1),"     YOLO night:",len(dangerous_yolo2),"     YOLO occlusion:",len(dangerous_yolo3),"    YOLO motion:",len(dangerous_yolo4))
print("YOLO+CNN dark:",len(dangerous_pipeline1),"YOLO+CNN night:",len(dangerous_pipeline2),"YOLO+CNN occlusion:",len(dangerous_pipeline3),"YOLO+CNN motion:",len(dangerous_pipeline4))
print("BOTH dark:",len(dangerous_both1),"     BOTH night:",len(dangerous_both2),"     BOTH occlusion:",len(dangerous_both3),"    BOTH motion:",len(dangerous_both4))


Krytyczne błędy, wpuszczenie drapieżnika
----------------
YOLO dark: 5      YOLO night: 6      YOLO occlusion: 4     YOLO motion: 5
YOLO+CNN dark: 4 YOLO+CNN night: 3 YOLO+CNN occlusion: 13 YOLO+CNN motion: 12
BOTH dark: 2      BOTH night: 2      BOTH occlusion: 3     BOTH motion: 1


In [89]:
locking_chicken_yolo = df[
    (df["ground_truth"]=="OPEN") & ((df["yolo"]=="CLOSE"))
]
locking_chicken_yolo1 = df_dark[
    (df_dark["ground_truth"]=="OPEN") & ((df_dark["yolo"]=="CLOSE"))
]
locking_chicken_yolo2 = df_night[
    (df_night["ground_truth"]=="OPEN") & ((df_night["yolo"]=="CLOSE"))
]
locking_chicken_yolo3 = df_occlusion[
    (df_occlusion["ground_truth"]=="OPEN") & ((df_occlusion["yolo"]=="CLOSE"))
]
locking_chicken_yolo4 = df_motion_blur[
    (df_motion_blur["ground_truth"]=="OPEN") & ((df_motion_blur["yolo"]=="CLOSE"))
]
locking_chicken_pipeline = df[
    (df["ground_truth"]=="OPEN") & ((df["pipeline"]=="CLOSE"))
]
locking_chicken_pipeline1 = df_dark[
    (df_dark["ground_truth"]=="OPEN") & ((df_dark["pipeline"]=="CLOSE"))
]
locking_chicken_pipeline2 = df_night[
    (df_night["ground_truth"]=="OPEN") & ((df_night["pipeline"]=="CLOSE"))
]
locking_chicken_pipeline3 = df_occlusion[
    (df_occlusion["ground_truth"]=="OPEN") & ((df_occlusion["pipeline"]=="CLOSE"))
]
locking_chicken_pipeline4 = df_motion_blur[
    (df_motion_blur["ground_truth"]=="OPEN") & ((df_motion_blur["pipeline"]=="CLOSE"))
]

locking_chicken_both = df[
    (df["ground_truth"]=="OPEN") & ((df["yolo"]=="CLOSE") | (df["pipeline"]=="CLOSE"))
]
locking_chicken_both1 = df_dark[
    (df_dark["ground_truth"]=="OPEN") & ((df_dark["yolo"]=="CLOSE") | (df_dark["pipeline"]=="CLOSE"))
]
locking_chicken_both2 = df_night[
    (df_night["ground_truth"]=="OPEN") & ((df_night["yolo"]=="CLOSE") | (df_night["pipeline"]=="CLOSE"))
]
locking_chicken_both3 = df_occlusion[
    (df_occlusion["ground_truth"]=="OPEN") & ((df_occlusion["yolo"]=="CLOSE") | (df_occlusion["pipeline"]=="CLOSE"))
]
locking_chicken_both4 = df_motion_blur[
    (df_motion_blur["ground_truth"]=="OPEN") & ((df_motion_blur["yolo"]=="CLOSE") | (df_motion_blur["pipeline"]=="CLOSE"))
]

In [ ]:
print()
print("Niekrytyczne błędy, niewpuszczenie kur")
print("----------------")
print("YOLO dark:",len(locking_chicken_both1),"     YOLO night:",len(locking_chicken_both2),"     YOLO occlusion:",len(locking_chicken_both3),"    YOLO motion:",len(locking_chicken_both4))



Niekrytyczne błędy, niewpuszczenie kur
----------------
YOLO dark: 26      YOLO night: 58      YOLO occlusion: 50     YOLO motion: 78


In [86]:
data = [[len(dangerous_yolo), len(dangerous_yolo1), len(dangerous_yolo2), len(dangerous_yolo3), len(dangerous_yolo4)],
        [len(dangerous_pipeline), len(dangerous_pipeline1), len(dangerous_pipeline2), len(dangerous_pipeline3), len(dangerous_pipeline4)],
        [len(dangerous_both), len(dangerous_both1), len(dangerous_both2), len(dangerous_both3), len(dangerous_both4)]]
columns = ["normal", "dark", "night", "occlusion", "motion"]
index = ["yolo", "yolo+cnn", "both"]
table = pd.DataFrame(data, columns=columns, index=index)
tolatextable = table.to_latex(index=True, float_format="{:.2f}".format,caption = "placeholder", label = "placeholder", position = "!h")
print("\\setlength{\\tabcolsep}{6pt}")
print("\\multicolumn{5}{c}{\\textbf{Krytyczne błędy, wpuszczenie przeciwnika}} ")
print(tolatextable)

\setlength{\tabcolsep}{6pt}
\multicolumn{5}{c}{\textbf{Krytyczne błędy, wpuszczenie przeciwnika}} 
\begin{table}[!h]
\caption{placeholder}
\label{placeholder}
\begin{tabular}{lrrrrr}
\toprule
 & normal & dark & night & occlusion & motion \\
\midrule
yolo & 4 & 5 & 6 & 4 & 5 \\
yolo+cnn & 7 & 4 & 3 & 13 & 12 \\
both & 2 & 2 & 2 & 3 & 1 \\
\bottomrule
\end{tabular}
\end{table}



In [93]:
data_locking = [[len(locking_chicken_yolo), len(locking_chicken_yolo1), len(locking_chicken_yolo2), len(locking_chicken_yolo3), len(locking_chicken_yolo4)],
    [len(locking_chicken_pipeline), len(locking_chicken_pipeline1), len(locking_chicken_pipeline2), len(locking_chicken_pipeline3), len(locking_chicken_pipeline4)],
    [len(locking_chicken_both), len(locking_chicken_both1), len(locking_chicken_both2), len(locking_chicken_both3), len(locking_chicken_both4)]]

columns_lock = ["normal", "dark", "night", "occlusion", "motion"]
index_lock = ["yolo", "yolo+cnn", "both"]

table2 = pd.DataFrame(data_locking, columns=columns, index=index)

tolatextable2 = table2.to_latex(index=True, float_format="{:.2f}".format,caption = "placeholder", label = "placeholder", position = "!h")
print("\\setlength{\\tabcolsep}{6pt}")
print("\\multicolumn{5}{c}{\\textbf{Niekrytycznie błędy, niewpuszczenie kur}} ")
print(tolatextable2)

\setlength{\tabcolsep}{6pt}
\multicolumn{5}{c}{\textbf{Niekrytycznie błędy, niewpuszczenie kur}} 
\begin{table}[!h]
\caption{placeholder}
\label{placeholder}
\begin{tabular}{lrrrrr}
\toprule
 & normal & dark & night & occlusion & motion \\
\midrule
yolo & 6 & 13 & 24 & 41 & 67 \\
yolo+cnn & 10 & 23 & 57 & 49 & 77 \\
both & 12 & 26 & 58 & 50 & 78 \\
\bottomrule
\end{tabular}
\end{table}



In [ ]:
latex_table = comparison.to_latex(index=True, float_format="{:.2f}".format,caption = "placeholder", label = "placeholder", position = "!h")
print("\\setlength{\\tabcolsep}{6pt}")
print(latex_table)

\setlength{\tabcolsep}{6pt}
\begin{table}[!h]
\caption{placeholder}
\label{placeholder}
\begin{tabular}{lrrrrrrrrr}
\toprule
 & Accuracy & Precision & Recall & F1 & TP & FP & TN & FN & Time \\
\midrule
YOLO normal & 0.99 & 0.99 & 0.99 & 0.99 & 481 & 4 & 387 & 6 & 14.26 \\
YOLO+CNN normal & 0.98 & 0.99 & 0.98 & 0.98 & 477 & 7 & 384 & 10 & 21.39 \\
YOLO dark & 0.98 & 0.99 & 0.97 & 0.98 & 474 & 5 & 386 & 13 & 11.75 \\
YOLO+CNN dark & 0.97 & 0.99 & 0.95 & 0.97 & 464 & 4 & 387 & 23 & 18.72 \\
YOLO night & 0.97 & 0.99 & 0.95 & 0.97 & 463 & 6 & 385 & 24 & 11.63 \\
YOLO+CNN night & 0.93 & 0.99 & 0.88 & 0.93 & 430 & 3 & 388 & 57 & 18.14 \\
YOLO occlusion & 0.95 & 0.99 & 0.92 & 0.95 & 446 & 4 & 387 & 41 & 12.44 \\
YOLO+CNN occlusion & 0.93 & 0.97 & 0.90 & 0.93 & 438 & 13 & 378 & 49 & 19.95 \\
YOLO motion & 0.92 & 0.99 & 0.86 & 0.92 & 420 & 5 & 386 & 67 & 11.84 \\
YOLO+CNN motion & 0.90 & 0.97 & 0.84 & 0.90 & 410 & 12 & 379 & 77 & 17.62 \\
\bottomrule
\end{tabular}
\end{table}



In [63]:
improved = df_dark[
    (df_dark["ground_truth"]=="CLOSE") &
    (df_dark["yolo"]=="OPEN") &
    (df_dark["pipeline"]=="CLOSE")
]
improved

,image,ground_truth,yolo,pipeline,yolo_time_ms,pipeline_time_ms,objects_gt,objects_yolo,objects_pipeline
327,Image-82-7d6856.jpg,CLOSE,OPEN,CLOSE,12.8715,16.2232,[1],[0],[1]
333,Image-84-bd2f1b.jpg,CLOSE,OPEN,CLOSE,9.7900,13.0119,[1],[0],[1]
836,raptor__gbif_raptor_00679_jpg.rf.IO9lWEg1PBD8V...,CLOSE,OPEN,CLOSE,12.5839,17.4623,[1],[0],[1]


In [64]:
improved = df_night[
    (df_night["ground_truth"]=="CLOSE") &
    (df_night["yolo"]=="OPEN") &
    (df_night["pipeline"]=="CLOSE")

]
improved

,image,ground_truth,yolo,pipeline,yolo_time_ms,pipeline_time_ms,objects_gt,objects_yolo,objects_pipeline
300,Image-62-e710b5.jpg,CLOSE,OPEN,CLOSE,12.4800,16.4372,[1],[0],[1]
327,Image-82-7d6856.jpg,CLOSE,OPEN,CLOSE,11.9616,15.9310,[1],[0],[1]
333,Image-84-bd2f1b.jpg,CLOSE,OPEN,CLOSE,8.9482,12.9065,[1],[0],[1]
836,raptor__gbif_raptor_00679_jpg.rf.IO9lWEg1PBD8V...,CLOSE,OPEN,CLOSE,12.3806,17.7643,[1],[0],[1]


In [65]:
improved = df_occlusion[
    (df_occlusion["ground_truth"]=="CLOSE") &
    (df_occlusion["yolo"]=="OPEN") &
    (df_occlusion["pipeline"]=="CLOSE")
]
improved

,image,ground_truth,yolo,pipeline,yolo_time_ms,pipeline_time_ms,objects_gt,objects_yolo,objects_pipeline
327,Image-82-7d6856.jpg,CLOSE,OPEN,CLOSE,14.3586,18.2968,[1],[0],[1]


In [66]:
improved = df_motion_blur[
    (df_motion_blur["ground_truth"]=="CLOSE") &
    (df_motion_blur["yolo"]=="OPEN") &
    (df_motion_blur["pipeline"]=="CLOSE")
]
improved

,image,ground_truth,yolo,pipeline,yolo_time_ms,pipeline_time_ms,objects_gt,objects_yolo,objects_pipeline
327,Image-82-7d6856.jpg,CLOSE,OPEN,CLOSE,12.7262,15.4884,[1],[0],[1]
836,raptor__gbif_raptor_00679_jpg.rf.IO9lWEg1PBD8V...,CLOSE,OPEN,CLOSE,13.1566,17.7518,[1],[0],[1]
872,raptor__raptor_012_jpg.rf.yRUSw4eHaOfcUqIhXQVM...,CLOSE,OPEN,CLOSE,12.9897,16.8520,"[1, 1]",[0],[1]
873,raptor__raptor_014_jpg.rf.WcLMSucKo0EGCfOoYu3y...,CLOSE,OPEN,CLOSE,12.5535,17.2942,"[1, 1]",[0],[1]


In [67]:
deproved = df_motion_blur[
    (df_motion_blur["ground_truth"]=="CLOSE") &
    (df_motion_blur["yolo"]=="CLOSE") &
    (df_motion_blur["pipeline"]=="OPEN")
]
deproved

,image,ground_truth,yolo,pipeline,yolo_time_ms,pipeline_time_ms,objects_gt,objects_yolo,objects_pipeline
89,coyote__lila_AMMonitor_Camera_Traps_MMP-Suc2_0...,CLOSE,CLOSE,OPEN,11.1983,15.3173,[1],[1],[0]
109,coyote__lila_Felidae_Conservation_Fund_2020-20...,CLOSE,CLOSE,OPEN,11.1235,13.9421,[1],[1],[0]
122,coyote__lila_Felidae_Conservation_Fund_2020-20...,CLOSE,CLOSE,OPEN,9.9021,15.9467,[1],[1],[0]
142,coyote__lila_WSU_Lynx_IMG_0965_jpg.rf.vqQg0Cpt...,CLOSE,CLOSE,OPEN,9.0695,14.7010,[1],[1],[0]
186,fox__gbif_fox_0454_jpg.rf.ET1jFfTjXFmpusKl9rJT...,CLOSE,CLOSE,OPEN,9.1617,15.6899,[1],[1],[0]
213,Image-10-95384c.jpg,CLOSE,CLOSE,OPEN,12.9854,18.0232,[1],[1],[0]
285,Image-51-eb7770.jpg,CLOSE,CLOSE,OPEN,11.5715,13.2655,[1],[1],[0]
332,Image-84-a5ad3f.jpg,CLOSE,CLOSE,OPEN,13.0238,21.0804,[1],[1],[0]
339,Image-88-8f30f6.jpg,CLOSE,CLOSE,OPEN,11.1404,13.3262,[1],[1],[0]
384,neg_people__people_007_jpg.rf.Oa8IhLxDK52JvmZ2...,CLOSE,CLOSE,OPEN,11.3546,16.2672,[],[1],[0]


In [68]:
empty_images0 = 0
false_detections0 = 0
for row in results_dark:

    if len(row["objects_gt"]) == 0:

        empty_images0 += 1

        if len(row["objects_yolo"]) > 0:
            false_detections0 += 1

print("Puste obrazy:",empty_images0)
print("Fałszywe detekcje:",false_detections0)
if empty_images0>0:

    print(
        "Odsetek:",
        false_detections0/empty_images0
    )

Puste obrazy: 37
Fałszywe detekcje: 2
Odsetek: 0.05405405405405406


In [69]:
empty_images1 = 0
false_detections1 = 0

for row in results_night:

    if len(row["objects_gt"]) == 0:

        empty_images1 += 1

        if len(row["objects_yolo"]) > 0:
            false_detections1 += 1

print("Puste obrazy:",empty_images1)
print("Fałszywe detekcje:",false_detections1)

if empty_images1>0:

    print(
        "Odsetek:",
        false_detections1/empty_images1
    )

Puste obrazy: 37
Fałszywe detekcje: 2
Odsetek: 0.05405405405405406


In [70]:
empty_images2 = 0
false_detections2 = 0

for row in results_occlusion:

    if len(row["objects_gt"]) == 0:

        empty_images2 += 1

        if len(row["objects_yolo"]) > 0:
            false_detections2 += 1

print("Puste obrazy:",empty_images2)
print("Fałszywe detekcje:",false_detections2)

if empty_images2>0:

    print(
        "Odsetek:",
        false_detections2/empty_images2
    )

Puste obrazy: 37
Fałszywe detekcje: 2
Odsetek: 0.05405405405405406


In [71]:
empty_images3 = 0
false_detections3 = 0

for row in results_motion_blur:

    if len(row["objects_gt"]) == 0:

        empty_images3 += 1

        if len(row["objects_yolo"]) > 0:
            false_detections3 += 1

print("Puste obrazy:",empty_images3)
print("Fałszywe detekcje:",false_detections3)

if empty_images3>0:

    print(
        "Odsetek:",
        false_detections3/empty_images3
    )

Puste obrazy: 37
Fałszywe detekcje: 2
Odsetek: 0.05405405405405406


In [72]:

#ZMIEŃ CONFIDENC POTEM NA 0,5 I PORÓWNAJ!!!!!!